# 🔬 The Hidden Math Behind "Cheap" Printers
## Monte Carlo Simulation Reveals the True Cost of Razor-and-Blade Pricing

**Author:** [Fabio Oliveira](https://www.linkedin.com/in/fabionoliveirastr/)  
**Published on:** [Towards Data Science](https://towardsdatascience.com/)  
**Date:** February 2025

---

This notebook simulates **10,000 five-year ownership scenarios** for five consumer products that use razor-and-blade pricing, modeling:

- **Stochastic usage patterns** (truncated normal distributions)
- **Heterogeneous consumer discount rates** calibrated to [Hausman (1979)](https://www.jstor.org/stable/3003415) — Beta(2, 7) ≈ mean 22%
- **Price variability** reflecting real Brazilian market conditions (Feb 2025)

The simulation quantifies the **"Myopia Tax"** — the gap between what consumers actually pay over 5 years and what they *perceive* they'll pay at the point of purchase, due to implicit discounting of future costs.

### Key Findings

| Product | Median 5yr TCO | TCO Multiplier | Myopia Tax |
|---------|---------------|---------------|------------|
| Nespresso Essenza Mini | R\$12,898 | 25.8× | R\$4,921 (38%) |
| PS5 Slim Digital | R\$12,816 | 4.4× | R\$3,792 (30%) |
| HP DeskJet 2874 | R\$5,155 | 14.3× | R\$1,890 (37%) |
| Gillette Fusion 5 | R\$4,575 | 101.7× | R\$1,808 (40%) |
| Epson EcoTank L3210 | R\$1,214 | 1.4× | R\$142 (12%) |

### Theoretical Framework

- **Shrouded Attributes** — Gabaix & Laibson (2006, *QJE*): market competition does not eliminate cost-hiding
- **Implicit Discount Rates** — Hausman (1979): consumers apply ~20-25% discount rates to future costs
- **Hold-Up Problem** — Williamson (1985, Nobel 2009): relationship-specific investment destroys bargaining power

## 1. Setup & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import truncnorm, beta, gaussian_kde
from pathlib import Path

# Reproducibility
np.random.seed(42)

# Simulation parameters
N_SIMULATIONS = 10_000
YEARS = 5

# Plot styling
sns.set_style("whitegrid")
plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

CATEGORY_COLORS = {
    'Printer': '#2196F3',
    'Coffee': '#795548',
    'Razor': '#4CAF50',
    'Gaming': '#9C27B0',
}

print(f"Configuration: {N_SIMULATIONS:,} simulations × {YEARS} years")
print(f"Total scenarios per product: {N_SIMULATIONS:,}")

## 2. Product Definitions

All prices in **BRL (R\$)**, sourced from Brazilian retail (Magazine Luiza, Amazon BR, HP Store, Nespresso BR — February 2025).

Each product is defined by:
- **Entry price**: the base product (printer, machine, handle, console)
- **Consumable unit price**: mean and std of per-unit consumable cost
- **Units per year**: mean and std of annual consumption rate
- **Lock-in strength**: qualitative score (0 = open system, 1 = full DRM/ecosystem lock)

In [ ]:
PRODUCTS = {
    'HP DeskJet 2874': {
        'entry_price': 360,
        'consumable_unit_price': {'mean': 160, 'std': 20},     # HP 667 combo cartridge
        'units_per_year': {'mean': 6, 'std': 2},               # cartridge replacements/yr
        'lock_in_strength': 0.85,                               # DRM + Dynamic Security
        'category': 'Printer',
        'short_name': 'HP DeskJet',
    },
    'Epson EcoTank L3210': {
        'entry_price': 850,
        'consumable_unit_price': {'mean': 56, 'std': 10},      # T544 ink bottle kit
        'units_per_year': {'mean': 1.3, 'std': 0.5},           # refills per year
        'lock_in_strength': 0.20,                               # open tank system
        'category': 'Printer',
        'short_name': 'Epson EcoTank',
    },
    'Nespresso Essenza Mini': {
        'entry_price': 500,
        'consumable_unit_price': {'mean': 3.40, 'std': 0.30},  # per capsule (official)
        'units_per_year': {'mean': 730, 'std': 180},            # ~2 cups/day ± variance
        'lock_in_strength': 0.95,                               # Vertuo barcode DRM
        'category': 'Coffee',
        'short_name': 'Nespresso',
    },
    'Gillette Fusion 5': {
        'entry_price': 45,
        'consumable_unit_price': {'mean': 17.50, 'std': 3.0},  # per cartridge
        'units_per_year': {'mean': 52, 'std': 12},              # weekly ± variance
        'lock_in_strength': 0.30,                               # design complexity only
        'category': 'Razor',
        'short_name': 'Gillette',
    },
    'PS5 Slim Digital': {
        'entry_price': 2900,
        'consumable_unit_price': {'mean': 280, 'std': 60},     # avg game price (PS Store BR)
        'units_per_year': {'mean': 7, 'std': 3},               # games/yr + PS Plus Essential
        'lock_in_strength': 0.90,                               # full ecosystem lock
        'category': 'Gaming',
        'short_name': 'PS5',
    },
}

# Quick summary
summary_rows = []
for name, p in PRODUCTS.items():
    det_tco = p['entry_price'] + p['consumable_unit_price']['mean'] * p['units_per_year']['mean'] * YEARS
    summary_rows.append({
        'Product': p['short_name'],
        'Entry Price': f"R\${p['entry_price']:,}",
        'Consumable/unit': f"R\${p['consumable_unit_price']['mean']:.2f}",
        'Units/year': f"{p['units_per_year']['mean']:.1f}",
        'Deterministic 5yr TCO': f"R\${det_tco:,.0f}",
        'Lock-in': f"{p['lock_in_strength']:.2f}",
    })

pd.DataFrame(summary_rows).set_index('Product')

## 3. Monte Carlo Simulation Engine

### Distribution Choices

| Variable | Distribution | Rationale |
|----------|-------------|-----------|
| **Usage rate** | Truncated Normal | Bounded below by 0; captures typical variation around habitual consumption |
| **Consumable price** | Truncated Normal | Market prices fluctuate within a band; no negative prices |
| **Consumer discount rate** | Beta(2, 7) | Right-skewed, mean ≈ 0.22; calibrated to Hausman (1979) finding of 20-25% |

### The "Myopia Gap" Concept

For each simulated consumer $i$ with implicit discount rate $r_i$:

$$\text{Actual TCO}_i = P_{\text{entry}} + \sum_{t=1}^{T} C_t \cdot U_t$$

$$\text{Perceived TCO}_i = P_{\text{entry}} + \sum_{t=1}^{T} \frac{C_t \cdot U_t}{(1 + r_i)^t}$$

$$\text{Myopia Gap}_i = \text{Actual TCO}_i - \text{Perceived TCO}_i$$

Where $C_t$ = consumable unit price at year $t$, $U_t$ = units consumed at year $t$.

In [ ]:
def truncnorm_rvs(mean, std, low_mult=0.3, high_mult=2.5, size=None):
    """Draw from truncated normal, preventing negative or extreme values."""
    a = (mean * low_mult - mean) / std
    b = (mean * high_mult - mean) / std
    return truncnorm.rvs(a, b, loc=mean, scale=std, size=size)


def simulate_tco(params, n_sims=N_SIMULATIONS, years=YEARS):
    """
    Monte Carlo simulation of Total Cost of Ownership.
    
    Models heterogeneous consumers with different implicit discount rates
    (Hausman 1979, Beta(2,7) -> mean ~0.22) and stochastic usage/pricing.
    
    Returns:
        actual_tco:    What consumers really pay over `years`
        perceived_tco: What consumers *think* they'll pay (discounted)
        discount_rates: Each consumer's implicit discount rate
        annual_costs:   (n_sims, years) matrix of annual consumable costs
    """
    entry = params['entry_price']
    price_p = params['consumable_unit_price']
    usage_p = params['units_per_year']

    # Consumer discount rates: Beta(2, 7) ~ mean 0.22, skewed right
    discount_rates = beta.rvs(2, 7, size=n_sims)

    # Stochastic prices and usage per year
    prices = truncnorm_rvs(price_p['mean'], price_p['std'],
                           low_mult=0.5, high_mult=1.8,
                           size=(n_sims, years))
    usage = truncnorm_rvs(usage_p['mean'], usage_p['std'],
                          low_mult=0.0, high_mult=2.5,
                          size=(n_sims, years))

    annual_costs = prices * usage

    # Actual TCO: entry + undiscounted consumable costs
    actual_tco = entry + annual_costs.sum(axis=1)

    # Perceived TCO: entry + discounted costs (consumer's biased view)
    perceived_tco = np.full(n_sims, float(entry))
    for yr in range(years):
        perceived_tco += annual_costs[:, yr] / (1 + discount_rates) ** (yr + 1)

    return actual_tco, perceived_tco, discount_rates, annual_costs

print("Simulation engine ready.")

## 4. Run Simulations

In [ ]:
# Run Monte Carlo for all products
results = {}

for name, params in PRODUCTS.items():
    actual, perceived, rates, annual = simulate_tco(params)
    results[name] = {
        'actual_tco': actual,
        'perceived_tco': perceived,
        'discount_rates': rates,
        'annual_costs': annual,
        'myopia_gap': actual - perceived,
        'params': params,
    }
    
    med = np.median(actual)
    p5, p95 = np.percentile(actual, [5, 95])
    gap = np.median(actual - perceived)
    mult = med / params['entry_price']
    
    print(f"{params['short_name']:>14s} | Median TCO: R${med:>8,.0f} | "
          f"90% CI: [{p5:,.0f} – {p95:,.0f}] | "
          f"Multiplier: {mult:>5.1f}× | Myopia Gap: R${gap:,.0f}")

print(f"\nTotal scenarios computed: {N_SIMULATIONS * len(PRODUCTS):,}")

## 5. Results & Visualizations

### 5.1 TCO Distribution per Product

Each histogram shows the distribution of 10,000 simulated 5-year TCO outcomes. The red dashed line marks the median; the shaded band covers the 90% confidence interval (5th to 95th percentile).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, (name, data) in enumerate(results.items()):
    ax = axes[idx]
    color = CATEGORY_COLORS[data['params']['category']]
    tco = data['actual_tco']

    ax.hist(tco, bins=60, density=True, alpha=0.65, color=color,
            edgecolor='white', linewidth=0.5)

    # KDE overlay
    kde = gaussian_kde(tco)
    x_range = np.linspace(tco.min() * 0.9, tco.max() * 1.1, 300)
    ax.plot(x_range, kde(x_range), color='black', linewidth=1.5, alpha=0.8)

    median = np.median(tco)
    p5, p95 = np.percentile(tco, [5, 95])

    ax.axvline(median, color='#E53935', linestyle='--', linewidth=2,
               label=f'Median: R${median:,.0f}')
    ax.axvspan(p5, p95, alpha=0.08, color='red',
               label=f'90% CI: R${p5:,.0f} – R${p95:,.0f}')

    ax.set_title(data['params']['short_name'], fontsize=13, fontweight='bold')
    ax.set_xlabel('5-Year TCO (R$)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8, loc='upper right')

axes[5].set_visible(False)
fig.suptitle('Monte Carlo TCO Distributions (n = 10,000 simulations per product)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('charts/01_tco_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Overlaid TCO Density Comparison

All five products on a single axis — showing how "cheap" entry-level products (HP DeskJet, Gillette) generate TCO distributions that overlap with or exceed "expensive" alternatives.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for name, data in results.items():
    tco = data['actual_tco']
    color = CATEGORY_COLORS[data['params']['category']]
    kde = gaussian_kde(tco)
    x = np.linspace(0, tco.max() * 1.15, 500)
    ax.fill_between(x, kde(x), alpha=0.25, color=color)
    ax.plot(x, kde(x), linewidth=2, color=color,
            label=f"{data['params']['short_name']} (median: R${np.median(tco):,.0f})")

ax.set_xlabel('5-Year Total Cost of Ownership (R$)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('TCO Density Comparison Across Product Categories',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.set_xlim(left=0)
plt.tight_layout()
plt.savefig('charts/02_overlaid_kde.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Box Plot Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

data_list = []
for name, data in results.items():
    df_temp = pd.DataFrame({
        'TCO': data['actual_tco'],
        'Product': data['params']['short_name'],
        'Category': data['params']['category'],
    })
    data_list.append(df_temp)
df = pd.concat(data_list, ignore_index=True)

order = sorted(results.keys(), key=lambda k: np.median(results[k]['actual_tco']))
order_short = [results[k]['params']['short_name'] for k in order]
palette = {results[k]['params']['short_name']: CATEGORY_COLORS[results[k]['params']['category']]
           for k in results}

sns.boxplot(data=df, x='Product', y='TCO', hue='Product', order=order_short,
            palette=palette, ax=ax, showfliers=False, width=0.6, legend=False)

for i, prod in enumerate(order_short):
    med = df[df['Product'] == prod]['TCO'].median()
    ax.text(i, med + 200, f'R${med:,.0f}', ha='center', fontsize=9, fontweight='bold', color='#333')

ax.set_ylabel('5-Year TCO (R$)', fontsize=12)
ax.set_xlabel('')
ax.set_title('TCO Distribution Comparison (outliers hidden)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/03_box_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 The "Myopia Tax"

The **Myopia Gap** = Actual TCO − Perceived TCO at point of purchase.

This quantifies how much consumers underestimate their 5-year spending due to implicit discounting of future costs (Hausman 1979). Higher gaps indicate products where the entry price is most deceptive relative to total ownership cost.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

names = [results[k]['params']['short_name'] for k in results]
gaps = [np.median(results[k]['myopia_gap']) for k in results]
colors_list = [CATEGORY_COLORS[results[k]['params']['category']] for k in results]

sorted_idx = np.argsort(gaps)
names = [names[i] for i in sorted_idx]
gaps = [gaps[i] for i in sorted_idx]
colors_list = [colors_list[i] for i in sorted_idx]

bars = ax.barh(names, gaps, color=colors_list, edgecolor='white', linewidth=1.5, height=0.6)

for bar, gap in zip(bars, gaps):
    ax.text(bar.get_width() + max(gaps) * 0.02, bar.get_y() + bar.get_height() / 2,
            f'R${gap:,.0f}', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Median "Myopia Gap" (R$)\nActual TCO − Perceived TCO at Purchase', fontsize=11)
ax.set_title('The "Myopia Tax": How Much More You Pay Than You Think\n'
             'Based on Hausman (1979) implicit discount rates, Beta(2,7)',
             fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('charts/04_myopia_tax.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Sensitivity Analysis (Tornado Diagram)

One-at-a-time sensitivity: vary each input ±1 standard deviation from its mean, holding all others at baseline. Shows which variable is the **dominant cost driver** for each product.

In [ ]:
targets = ['Nespresso Essenza Mini', 'HP DeskJet 2874', 'Gillette Fusion 5']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, prod_name in zip(axes, targets):
    p = PRODUCTS[prod_name]
    baseline = p['entry_price'] + (p['consumable_unit_price']['mean']
                                   * p['units_per_year']['mean'] * YEARS)
    factors = {}

    # Usage +/- 1 SD
    lo_u = p['entry_price'] + p['consumable_unit_price']['mean'] * (
        p['units_per_year']['mean'] - p['units_per_year']['std']) * YEARS
    hi_u = p['entry_price'] + p['consumable_unit_price']['mean'] * (
        p['units_per_year']['mean'] + p['units_per_year']['std']) * YEARS
    factors['Usage Rate'] = (lo_u - baseline, hi_u - baseline)

    # Price +/- 1 SD
    lo_p = p['entry_price'] + (
        p['consumable_unit_price']['mean'] - p['consumable_unit_price']['std']
    ) * p['units_per_year']['mean'] * YEARS
    hi_p = p['entry_price'] + (
        p['consumable_unit_price']['mean'] + p['consumable_unit_price']['std']
    ) * p['units_per_year']['mean'] * YEARS
    factors['Consumable Price'] = (lo_p - baseline, hi_p - baseline)

    # Discount rate: 10% vs 35% (perceived TCO)
    pv_lo = p['entry_price'] + sum(
        p['consumable_unit_price']['mean'] * p['units_per_year']['mean']
        / (1.10) ** (y + 1) for y in range(YEARS))
    pv_hi = p['entry_price'] + sum(
        p['consumable_unit_price']['mean'] * p['units_per_year']['mean']
        / (1.35) ** (y + 1) for y in range(YEARS))
    factors['Discount Rate\n(Perceived)'] = (pv_hi - baseline, pv_lo - baseline)

    sorted_factors = sorted(factors.items(), key=lambda x: abs(x[1][1] - x[1][0]), reverse=True)

    for i, (label, (lo, hi)) in enumerate(sorted_factors):
        color = CATEGORY_COLORS[p['category']]
        ax.barh(i, hi, height=0.5, color=color, alpha=0.8, edgecolor='white')
        ax.barh(i, lo, height=0.5, color=color, alpha=0.5, edgecolor='white')

    ax.set_yticks(range(len(sorted_factors)))
    ax.set_yticklabels([f[0] for f in sorted_factors], fontsize=10)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('Change from Baseline TCO (R$)')
    ax.set_title(PRODUCTS[prod_name]['short_name'], fontsize=12, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Sensitivity Analysis: Which Variable Drives TCO Most?\n'
             '(±1 Standard Deviation from Mean)',
             fontsize=14, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig('charts/05_tornado.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.6 Convergence Plot

Verifies that 10,000 simulations provide stable estimates. The running mean of TCO for each product stabilizes by approximately 2,000 iterations — using 10,000 provides a 5× safety margin.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

checkpoints = np.arange(50, N_SIMULATIONS + 1, 50)

for name, data in results.items():
    color = CATEGORY_COLORS[data['params']['category']]
    running_means = [data['actual_tco'][:n].mean() for n in checkpoints]
    ax.plot(checkpoints, running_means, color=color, linewidth=1.5,
            label=data['params']['short_name'], alpha=0.85)

ax.set_xlabel('Number of Simulations', fontsize=12)
ax.set_ylabel('Running Mean TCO (R$)', fontsize=12)
ax.set_title('Convergence: Mean TCO Stabilizes by ~2,000 Iterations\n'
             '(10,000 used for robustness)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('charts/06_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.7 Lock-In Strength vs. TCO Multiplier

Maps each product on two dimensions:
- **X-axis**: Lock-in strength (0 = open system, 1 = full DRM/ecosystem lock)
- **Y-axis**: TCO multiplier (median 5-year TCO ÷ entry price)

Note: Gillette sits in the "disruption target" quadrant (low lock-in, high multiplier) — which is exactly why Dollar Shave Club succeeded.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for name, data in results.items():
    lock_in = data['params']['lock_in_strength']
    multiplier = np.median(data['actual_tco']) / data['params']['entry_price']
    color = CATEGORY_COLORS[data['params']['category']]

    ax.scatter(lock_in, multiplier, s=250, c=color,
               edgecolors='black', linewidth=1.2, zorder=5)
    ax.annotate(data['params']['short_name'], (lock_in, multiplier),
                textcoords="offset points", xytext=(12, 8),
                fontsize=11, fontweight='bold',
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.8))

# Quadrant labels
ax.axhline(y=15, color='gray', linestyle=':', alpha=0.4)
ax.axvline(x=0.55, color='gray', linestyle=':', alpha=0.4)
ax.text(0.15, 1.5, 'LOW lock-in\nLOW multiplier\n("Honest" products)',
        fontsize=8, color='gray', ha='center', style='italic')
ax.text(0.85, 1.5, 'HIGH lock-in\nLOW multiplier\n(Ecosystem play)',
        fontsize=8, color='gray', ha='center', style='italic')
ax.text(0.15, 80, 'LOW lock-in\nHIGH multiplier\n(Disruption target)',
        fontsize=8, color='gray', ha='center', style='italic')
ax.text(0.85, 80, 'HIGH lock-in\nHIGH multiplier\n(Maximum extraction)',
        fontsize=8, color='gray', ha='center', style='italic')

ax.set_xlabel('Lock-In Strength\n(0 = open system, 1 = full DRM/ecosystem lock)', fontsize=12)
ax.set_ylabel('TCO Multiplier\n(Median 5-Year TCO / Entry Price)', fontsize=12)
ax.set_title('The Lock-In vs. TCO Multiplier Map', fontsize=13, fontweight='bold')
ax.set_xlim(-0.05, 1.05)
ax.set_yscale('log')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('charts/07_lockin_multiplier.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.8 Exceedance Probability

Answers the question: **"What are the chances I'll pay more than X reais over 5 years?"**

The square markers show the 90th percentile — there's a 10% chance TCO will exceed this amount.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

targets = ['Nespresso Essenza Mini', 'HP DeskJet 2874', 'PS5 Slim Digital']

for name in targets:
    data = results[name]
    color = CATEGORY_COLORS[data['params']['category']]
    sorted_tco = np.sort(data['actual_tco'])
    cdf = np.arange(1, len(sorted_tco) + 1) / len(sorted_tco)

    ax.plot(sorted_tco, 1 - cdf, color=color, linewidth=2,
            label=data['params']['short_name'])

    p50 = np.percentile(data['actual_tco'], 50)
    p90 = np.percentile(data['actual_tco'], 90)
    ax.plot(p50, 0.50, 'o', color=color, markersize=8, zorder=5)
    ax.plot(p90, 0.10, 's', color=color, markersize=8, zorder=5)
    ax.annotate(f'R${p90:,.0f}', (p90, 0.10),
                textcoords="offset points", xytext=(8, 5),
                fontsize=9, color=color, fontweight='bold')

ax.set_xlabel('5-Year TCO (R$)', fontsize=12)
ax.set_ylabel('Probability of Exceeding This Cost', fontsize=12)
ax.set_title('Exceedance Probability: "What Are the Chances I\'ll Pay More Than X?"',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.02)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('charts/08_cumulative_probability.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary Statistics

In [ ]:
rows = []
for name, data in results.items():
    tco = data['actual_tco']
    perceived = data['perceived_tco']
    gap = data['myopia_gap']
    entry = data['params']['entry_price']
    
    rows.append({
        'Product': data['params']['short_name'],
        'Category': data['params']['category'],
        'Entry Price (R$)': entry,
        'Median TCO (R$)': round(np.median(tco)),
        '5th Pctl': round(np.percentile(tco, 5)),
        '95th Pctl': round(np.percentile(tco, 95)),
        'TCO Multiplier': round(np.median(tco) / entry, 1),
        'Median Perceived TCO': round(np.median(perceived)),
        'Myopia Gap (R$)': round(np.median(gap)),
        'Myopia Gap %': round(np.median(gap) / np.median(tco) * 100, 1),
        'Lock-In': data['params']['lock_in_strength'],
    })

summary_df = pd.DataFrame(rows).sort_values('Median TCO (R$)', ascending=False)
summary_df.set_index('Product')

## 7. References

1. **Gabaix, X. & Laibson, D.** (2006). "Shrouded Attributes, Consumer Myopia, and Information Suppression in Competitive Markets." *Quarterly Journal of Economics*, 121(2), 505–540. [DOI](https://doi.org/10.1162/qjec.2006.121.2.505)

2. **Hausman, J.** (1979). "Individual Discount Rates and the Purchase and Utilization of Energy-Using Durables." *Bell Journal of Economics*, 10(1), 33–54. [JSTOR](https://www.jstor.org/stable/3003415)

3. **Allcott, H. & Wozny, N.** (2014). "Gasoline Prices, Fuel Economy, and the Energy Paradox." *Review of Economics and Statistics*, 96(5), 779–795.

4. **Williamson, O.** (1985). *The Economic Institutions of Capitalism*. Free Press.

5. **Morwitz, V., Greenleaf, E. & Johnson, E.** (1998). "Divide and Prosper: Consumers' Reactions to Partitioned Prices." *Journal of Marketing Research*, 35(4), 453–463.

---

**License:** MIT  
**Author:** [Fabio Oliveira](https://www.linkedin.com/in/fabionoliveirastr/) — Electronic & Computer Engineer (UFRJ), Strategy & Operations Professional  
**Companion Article:** [Towards Data Science](https://towardsdatascience.com/)